# SimCLR Ablation Study — Session A: Single-Augmentation Ablations
Remove one augmentation at a time; all others remain as in the baseline.

| Experiment | What changes |
|---|---|
| `ablation_no_crop` | RandomResizedCrop → Resize |
| `ablation_no_flip` | HorizontalFlip off |
| `ablation_no_jitter` | ColorJitter off |
| `ablation_no_grayscale` | RandomGrayscale off |
| `ablation_no_blur` | GaussianBlur off |

**Expected runtime (Colab T4):** ~10–12 min × 5 = ~1h  
**Before running:** Runtime → Change runtime type → T4 GPU

> **Tip — save checkpoints across sessions:** Run the *Mount Google Drive* cell below.  
> If you skip it, checkpoints save to `/content` and are lost when the session ends.

In [ ]:
# ── Optional: Mount Google Drive to persist checkpoints ──────────────────────
# Skip this cell if you don't need to save between sessions.
MOUNT_DRIVE = True   # ← set False to skip

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/simclr_ablations'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
if torch.cuda.is_available():
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'No GPU detected — Runtime → Change runtime type → T4 GPU'

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/SAlaMusa/IDL.git'
REPO_DIR = '/content/IDL'

# OUTPUT_DIR already set in Drive cell; fall back to /content if Drive not mounted
if 'OUTPUT_DIR' not in dir():
    OUTPUT_DIR = '/content'
    os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
import sys, glob, shutil, csv, datetime
sys.path.insert(0, REPO_DIR)
from models.resnet_simclr import ResNetSimCLR
print('Imports OK')

In [ ]:
RESULTS_CSV = os.path.join(OUTPUT_DIR, 'session_a_results.csv')

if not os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, 'w', newline='') as f:
        csv.writer(f).writerow(['exp_name', 'config', 'best_top1', 'checkpoint', 'timestamp'])


def run_experiment(exp_name, config_path, epochs=100, batch=256):
    """Pretrain + linear-eval one ablation. Skips pretraining if checkpoint exists."""
    ckpt_name = f'{exp_name}_ep{epochs}.pth.tar'
    ckpt_path = os.path.join(OUTPUT_DIR, ckpt_name)

    # ── Pretraining ────────────────────────────────────────────────────────────
    if os.path.exists(ckpt_path):
        print(f'[{exp_name}] Checkpoint exists — skipping pretraining.')
    else:
        print(f"\n{'='*60}\n[{exp_name}] PRETRAINING ({epochs} epochs)\n{'='*60}")
        result = subprocess.run([
            'python', 'run.py',
            '--config',     config_path,
            '--epochs',     str(epochs),
            '--batch-size', str(batch),
            '-j',           '4',          # Colab gives 2 vCPUs; 4 workers saturates them
            '--seed',       '42',
            '--fp16-precision',
        ])
        if result.returncode != 0:
            raise RuntimeError(f'Pretraining failed for {exp_name}.')
        latest = max(glob.glob('runs/**/*.pth.tar', recursive=True), key=os.path.getmtime)
        shutil.copy2(latest, ckpt_path)
        print(f'[{exp_name}] Checkpoint saved → {ckpt_path}')

    # ── Linear evaluation ──────────────────────────────────────────────────────
    print(f'\n[{exp_name}] LINEAR EVALUATION')
    eval_result = subprocess.run([
        'python', 'linear_eval.py',
        '--checkpoint', ckpt_path,
        '--dataset',    'cifar10',
        '--arch',       'resnet18',
        '--epochs',     '30',         # 100→30: ~70% faster, ~0.5% accuracy cost
        '-b',           '512',        # larger batch = fewer steps per epoch
        '-j',           '4',
        '--seed',       '42',
    ])
    if eval_result.returncode != 0:
        raise RuntimeError(f'Linear eval failed for {exp_name}.')

    with open('linear_eval_results.csv') as f:
        best_top1 = float(list(csv.DictReader(f))[-1]['best_top1'])

    with open(RESULTS_CSV, 'a', newline='') as f:
        csv.writer(f).writerow([
            exp_name, config_path, f'{best_top1:.2f}', ckpt_path,
            datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
        ])
    print(f'[{exp_name}] Best Top-1: {best_top1:.2f}%')
    return best_top1


print('Helper defined.')

## Run Ablations

In [ ]:
ablations = [
    ('ablation_no_crop',      'configs/ablation_no_crop.yaml'),
    ('ablation_no_flip',      'configs/ablation_no_flip.yaml'),
    ('ablation_no_jitter',    'configs/ablation_no_jitter.yaml'),
    ('ablation_no_grayscale', 'configs/ablation_no_grayscale.yaml'),
    ('ablation_no_blur',      'configs/ablation_no_blur.yaml'),
]

results = {}
for exp_name, config_path in ablations:
    results[exp_name] = run_experiment(exp_name, config_path)

print('\n=== Session A Complete ===')
for name, acc in results.items():
    print(f'  {name:<30} {acc:.2f}%')
print(f'\nResults saved to: {RESULTS_CSV}')

In [ ]:
# ── Download results CSV to your local machine ────────────────────────────────
# Only needed if you did NOT mount Google Drive above.
from google.colab import files
files.download(RESULTS_CSV)